# **ENSO Phase Lifetime Experiment Split by Eruption Seasonality - Reconstructions**

Outputs: 

1) Per-eruption raw remaining lifetime CSV
2) Control (non-eruption) remaining lifetime CSV

In [1]:
import os
import numpy as np
import pandas as pd


# USER SETTINGS

FILE_PATH = "/home/563/ft3359/ENSO_Records_all.csv"
YEAR_COL  = "Years"

RECON_COLUMNS = [
    "Zhu et al. (2022) Li13b6.",
    "Wilson et al. (2010) Nino34 COAPCR",
    "Freund et al. (2019) Nino4 DJF",
    "Li et al. (2011) NADA PC1",
    "Stahle et al. (1993)",
    "DArrigo et al. (2005) Nino3",
    "Datwyler et al. (2020) ENSO DJF",
    "Geay et al. (2013) Nino3",
    "Liu et al. (2024) PCR",
]

# ERUPTION LIST: all 20 eruptions (850-1849 CE) with known seasonality.

ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW,
    columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)

ERUPTION_YEARS = eruptions_df["yearCE"].to_numpy(dtype=int)

# Season classification

def classify_season(month: float) -> str:
    m = int(month)
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    if m in (9, 10, 11):
        return "SON"
    return None

eruptions_df["season"] = eruptions_df["month"].apply(classify_season)
SEASON_BY_YEAR = dict(zip(eruptions_df["yearCE"].astype(int), eruptions_df["season"]))

print(eruptions_df["season"].value_counts())

THRESH        = 0.5
EXCLUDE_YEARS = 5
OFFSET        = 0

N_MIN_PER_CELL = 1

PHASE_LABELS = {1: "El Niño", 0: "Neutral", -1: "La Niña"}
PHASE_CODES  = [1, 0, -1]

SAVE_CSV  = True
OUT_RAW_PATH  = "/home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/Seasonal_Analysis/Seasonal_raw_recon.csv"
OUT_CTRL_PATH = "/home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/Seasonal_Analysis/Seasonal_control_recon.csv"

# HELPERS

def clean_name(s: str) -> str:
    if s is None:
        return s
    s = str(s).strip()
    while len(s) > 0 and s[0] in ["'", '"']:
        s = s[1:].lstrip()
    while len(s) > 0 and s[-1] in ["'", '"']:
        s = s[:-1].rstrip()
    return s.strip()

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

def classify_phases(values: np.ndarray, thresh: float = 0.5) -> np.ndarray:
    x = np.asarray(values, dtype=float)
    ph = np.zeros_like(x, dtype=int)
    ph[x >=  thresh] =  1
    ph[x <= -thresh] = -1
    return ph

def remaining_lifetime(phases: np.ndarray) -> np.ndarray:
    phases = np.asarray(phases, dtype=int)
    n = len(phases)
    rem = np.zeros(n, dtype=int)
    i = 0
    while i < n:
        j = i
        while (j + 1 < n) and (phases[j + 1] == phases[i]):
            j += 1
        for k in range(i, j + 1):
            rem[k] = (j - k + 1)
        i = j + 1
    return rem

def eligible_control_mask(years: np.ndarray, eruption_years: np.ndarray, exclude_years: int) -> np.ndarray:
    yrs = np.asarray(years, dtype=int)
    eru = np.asarray(sorted(set(int(y) for y in eruption_years)), dtype=int)
    mask = np.ones_like(yrs, dtype=bool)
    mask &= ~np.isin(yrs, eru)
    for ey in eru:
        mask &= ~((yrs >= ey - exclude_years) & (yrs <= ey + exclude_years))
    return mask

# LOAD DATA

df_raw = pd.read_csv(FILE_PATH)
df = clean_columns(df_raw)

YEAR_COL_CLEAN = clean_name(YEAR_COL)
RECON_COLUMNS_CLEAN = [clean_name(x) for x in RECON_COLUMNS]

if YEAR_COL_CLEAN not in df.columns:
    raise KeyError(
        f"YEAR_COL='{YEAR_COL_CLEAN}' not found. "
        f"Example columns: {list(df.columns)[:25]}"
    )

# BUILD RAW + CONTROL ROWS

raw_rows  = []
ctrl_rows = []

for recon in RECON_COLUMNS_CLEAN:
    if recon not in df.columns:
        print(f"Skipping '{recon}' (missing column).")
        continue

    # extract year and value
    tmp = pd.DataFrame({
        "year":  pd.to_numeric(df[YEAR_COL_CLEAN], errors="coerce"),
        "value": pd.to_numeric(df[recon],           errors="coerce"),
    }).dropna().sort_values("year")

    years     = tmp["year"].astype(int).values
    vals      = tmp["value"].astype(float).values
    phases    = classify_phases(vals, THRESH)
    lifetimes = remaining_lifetime(phases)

    year_to_idx = {int(y): i for i, y in enumerate(years)}

    for phc in PHASE_CODES:
        ph_label = PHASE_LABELS[phc]

        # RAW
        for ey in ERUPTION_YEARS:
            ay = int(ey) + OFFSET
            if ay not in year_to_idx:
                continue
            idx = year_to_idx[ay]
            if phases[idx] != phc:
                continue
            lt = float(lifetimes[idx])
            if not np.isfinite(lt):
                continue
            raw_rows.append({
                "source":             "Reconstructions",
                "reconstruction":     recon,
                "Season":             SEASON_BY_YEAR.get(int(ey)),
                "Phase":              ph_label,
                "eruption_year":      int(ey),
                "remaining_lifetime": lt,
            })

        # CONTROL — phase-matched
        ok_ctrl   = eligible_control_mask(years, ERUPTION_YEARS, EXCLUDE_YEARS)
        ctrl_idxs = [year_to_idx[int(y)] for y in years[(phases == phc) & ok_ctrl]]
        for idx in ctrl_idxs:
            lt = float(lifetimes[idx])
            if not np.isfinite(lt):
                continue
            ctrl_rows.append({
                "source":             "Reconstructions",
                "reconstruction":     recon,
                "Phase":              ph_label,
                "remaining_lifetime": lt,
            })

df_out_raw  = pd.DataFrame(raw_rows)
df_out_ctrl = pd.DataFrame(ctrl_rows)

phase_order = ["El Niño", "Neutral", "La Niña"]
season_order = ["DJF", "MAM", "JJA", "SON"]

if not df_out_raw.empty:
    df_out_raw["Phase"] = pd.Categorical(df_out_raw["Phase"], categories=phase_order, ordered=True)
    df_out_raw["Season"] = pd.Categorical(df_out_raw["Season"], categories=season_order, ordered=True)
if not df_out_ctrl.empty:
    df_out_ctrl["Phase"] = pd.Categorical(df_out_ctrl["Phase"], categories=phase_order, ordered=True)

df_out_raw  = df_out_raw.sort_values(["Season", "Phase", "reconstruction", "eruption_year"]).reset_index(drop=True)
df_out_ctrl = df_out_ctrl.sort_values(["Phase", "reconstruction"]).reset_index(drop=True)


# SAVE

if SAVE_CSV:
    os.makedirs(os.path.dirname(OUT_RAW_PATH), exist_ok=True)
    df_out_raw.to_csv(OUT_RAW_PATH,   index=False)
    df_out_ctrl.to_csv(OUT_CTRL_PATH, index=False)
    print(f"Saved raw lifetimes : {OUT_RAW_PATH}  ({len(df_out_raw)} rows, {df_out_raw['reconstruction'].nunique()} reconstructions)")
    print(f"Saved ctrl lifetimes: {OUT_CTRL_PATH}  ({len(df_out_ctrl)} rows, {df_out_ctrl['reconstruction'].nunique()} reconstructions)")

season
MAM    6
DJF    6
SON    4
JJA    4
Name: count, dtype: int64
Saved raw lifetimes : /home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/Seasonal_Analysis/Seasonal_raw_recon.csv  (142 rows, 9 reconstructions)
Saved ctrl lifetimes: /home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/Seasonal_Analysis/Seasonal_control_recon.csv  (4831 rows, 9 reconstructions)
